# Multi-step addition: 10,000 position-matched TopK extension

This post-report extension trains TopK-256 SAEs on 10,000 balanced addition prompts captured at the exact teacher-forced position used to predict the tens digit. It repeats both attribution-graph feature screening and output-digit-balanced carry localisation. All artifacts use a new namespace and cannot overwrite the original mathematics experiments.

## 1. Mount Drive, fetch the repository, and install it

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, shlex, subprocess, sys
from pathlib import Path

def run_cmd(command):
    command = [str(part) for part in command]
    print('$', shlex.join(command), flush=True)
    environment = os.environ.copy()
    environment['PYTHONUNBUFFERED'] = '1'
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=environment)
    if process.stdout is None:
        raise RuntimeError('Could not capture child-process output')
    for line in process.stdout:
        print(line, end='', flush=True)
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)

repo_url = 'https://github.com/evey-dev/test_run.git'
checkout = Path('/content/test_run')
if (checkout / '.git').is_dir():
    run_cmd(['git', '-C', checkout, 'pull', '--ff-only'])
else:
    if checkout.exists() and any(checkout.iterdir()):
        checkout = Path('/content/test_run_github')
    if (checkout / '.git').is_dir():
        run_cmd(['git', '-C', checkout, 'pull', '--ff-only'])
    else:
        run_cmd(['git', 'clone', '--depth', '1', repo_url, checkout])
os.chdir(checkout)
run_cmd([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.51,<5', 'accelerate', 'pandas<2.4', 'pyyaml', 'tqdm'])
run_cmd([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'])
print('Repository root:', Path.cwd())

## 2. Fixed design and persistent artifact paths

In [ ]:
import json, re, shutil
import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import display

ROOT = Path.cwd().resolve()
LAYERS = [4, 8, 12, 16, 20, 24, 28]
DATA_ARG = 'data/addition_large_10000.csv'
CONFIG_ARG = 'configs/sae_math_large_10000_topk256_config.yaml'
ACTIVATION_DIR = ROOT / 'mechanistic_data_math_large_10000'
CHECKPOINT_DIR = ROOT / 'mechanistic_data/sae_checkpoints_math_large_10000_topk256'
LOCAL_OUTPUT = ROOT / 'outputs/math_large_data_test'
DRIVE_ROOT = Path('/content/drive/MyDrive/mphil-project')
DRIVE_EXPERIMENT = DRIVE_ROOT / 'mechanistic_data/math_large_data_test'
DRIVE_ACTIVATIONS = DRIVE_EXPERIMENT / 'activations'
DRIVE_CHECKPOINTS = DRIVE_EXPERIMENT / 'checkpoints_topk256'
DRIVE_OUTPUT = DRIVE_ROOT / 'outputs/math_large_data_test'
for path in (LOCAL_OUTPUT, DRIVE_EXPERIMENT, DRIVE_ACTIVATIONS, DRIVE_CHECKPOINTS, DRIVE_OUTPUT):
    path.mkdir(parents=True, exist_ok=True)

required = [Path('data/generate_large_math_dataset.py'), Path(CONFIG_ARG), Path('src/math_carry_balanced_localization.py'), Path('src/math_carry_feature_screen.py')]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('Push the new experiment files to GitHub before running: ' + ', '.join(missing))
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime before continuing')
config = yaml.safe_load(Path(CONFIG_ARG).read_text())
assert config['activation_type'] == 'topk' and config['top_k'] == 256
assert config['layers'] == LAYERS and config['seed'] == 787 and config.get('save_latents') is False
print('GPU:', torch.cuda.get_device_name(0))
print('Drive experiment root:', DRIVE_EXPERIMENT)

## 3. Generate and audit the position-matched corpus

Each prompt ends after the teacher-forced hundreds digit `1`; its final-token MLP activation is therefore the state from which the model predicts the tens digit. Carry and no-carry cases are exactly balanced. Operand pairs used by the original `58+83` experiments are explicitly reserved.

In [ ]:
run_cmd([sys.executable, 'data/generate_large_math_dataset.py', '--count', '10000', '--seed', '787', '--output', DATA_ARG])
corpus = pd.read_csv(DATA_ARG)
assert len(corpus) == 10_000 and corpus['sentence'].nunique() == 10_000
assert corpus.groupby('IsCarry').size().to_dict() == {0: 5000, 1: 5000}
assert set(corpus['TeacherForcedPrefix']) == {1}
training_pairs = {(int(row.Operand1), int(row.Operand2)) for row in corpus.itertuples()}
for pair in [(58, 83), (83, 58), (44, 83), (83, 44), (54, 83), (83, 54), (32, 42), (42, 32)]:
    assert pair not in training_pairs
from src.heldout_validation import generated_math_cases
fresh_cases = generated_math_cases(140, Path(DATA_ARG), seed=4787)
assert len(fresh_cases) == 140 and all(case['absent_from_sae_corpus'] for case in fresh_cases)
display(corpus.groupby(['IsCarry', 'OutputTensDigit']).size().unstack(fill_value=0))
display(corpus[['Operand1', 'Operand2', 'IsCarry', 'OutputTensDigit', 'sentence']].head())

## 4. Capture final-token MLP activations at seven selected layers

In [ ]:
activation_files = [f'activations_layer{layer}.npy' for layer in LAYERS] + ['train_indices.npy', 'val_indices.npy', 'train_val_indices_per_layer.npy', 'activation_metadata.csv']
def complete(directory, names):
    return all((directory / name).exists() for name in names)

if not complete(ACTIVATION_DIR, activation_files) and complete(DRIVE_ACTIVATIONS, activation_files):
    print('Restoring activation capture from Drive...')
    shutil.copytree(DRIVE_ACTIVATIONS, ACTIVATION_DIR, dirs_exist_ok=True)
if not complete(ACTIVATION_DIR, activation_files):
    run_cmd([sys.executable, '-m', 'src.capture_activations', '--model-config', 'configs/model_config.yaml', '--output-dir', str(ACTIVATION_DIR), '--prompt-csv', DATA_ARG, '--prompt-behaviour', 'math_carry_position_large_10000', '--layers', *map(str, LAYERS), '--seed', '787'])
    shutil.copytree(ACTIVATION_DIR, DRIVE_ACTIVATIONS, dirs_exist_ok=True)
else:
    print('Activation capture already complete; skipping capture.')
assert complete(ACTIVATION_DIR, activation_files)
metadata = pd.read_csv(ACTIVATION_DIR / 'activation_metadata.csv')
assert set(metadata['token'].astype(str).str.strip()) == {'1'}
train_indices = np.load(ACTIVATION_DIR / 'train_indices.npy')
val_indices = np.load(ACTIVATION_DIR / 'val_indices.npy')
assert len(train_indices) == 8000 and len(val_indices) == 2000
display(pd.DataFrame({'train': corpus.iloc[train_indices].groupby('IsCarry').size(), 'validation': corpus.iloc[val_indices].groupby('IsCarry').size()}))

## 5. Train or restore the 10k TopK-256 SAEs

In [ ]:
run_cmd([sys.executable, '-m', 'src.train', '--config', CONFIG_ARG, '--drive-dir', str(DRIVE_CHECKPOINTS)])
checkpoint_files = [name for layer in LAYERS for name in (f'sae_layer{layer}.pt', f'sae_layer{layer}_metadata.json')]
assert complete(CHECKPOINT_DIR, checkpoint_files)
print('Complete checkpoint layers:', LAYERS)
print('Dense latent arrays written:', len(list(CHECKPOINT_DIR.glob('latents_layer*.npy'))))

## 6. Measure held-out reconstruction and sparsity

In [ ]:
diagnostic_json = LOCAL_OUTPUT / 'math_large_10000_topk256_diagnostics.json'
diagnostic_csv = LOCAL_OUTPUT / 'math_large_10000_topk256_diagnostics.csv'
if (DRIVE_OUTPUT / diagnostic_json.name).exists() and (DRIVE_OUTPUT / diagnostic_csv.name).exists():
    shutil.copy2(DRIVE_OUTPUT / diagnostic_json.name, diagnostic_json)
    shutil.copy2(DRIVE_OUTPUT / diagnostic_csv.name, diagnostic_csv)
else:
    run_cmd([sys.executable, '-m', 'src.sae_diagnostics', '--config', CONFIG_ARG, '--label', 'math_large_10000_topk256', '--device', 'cuda', '--output-json', str(diagnostic_json), '--output-csv', str(diagnostic_csv)])
    shutil.copy2(diagnostic_json, DRIVE_OUTPUT / diagnostic_json.name)
    shutil.copy2(diagnostic_csv, DRIVE_OUTPUT / diagnostic_csv.name)
display(pd.read_csv(diagnostic_csv)[['layer', 'validation_fraction_variance_explained', 'validation_mean_relative_l2_error', 'validation_mean_l0', 'combined_dead_feature_fraction']])

## 7. Rebuild the `58+83` carry contrast graph

In [ ]:
GRAPH_STEM = 'math_large_10000_topk256_carry_58_83_4v3_graph'
graph_paths = {suffix: LOCAL_OUTPUT / f'{GRAPH_STEM}.{suffix}' for suffix in ('json', 'html', 'md')}
drive_graph_paths = {suffix: DRIVE_OUTPUT / path.name for suffix, path in graph_paths.items()}
if all(path.exists() for path in drive_graph_paths.values()):
    for suffix in graph_paths:
        shutil.copy2(drive_graph_paths[suffix], graph_paths[suffix])
else:
    run_cmd([sys.executable, '-m', 'src.attribution_graph', '--prompt', 'Question: What is 58 + 83? Answer: 1', '--target', '4', '--contrast-target', '3', '--layers', *map(str, LAYERS), '--sae-config', CONFIG_ARG, '--top-k-nodes', '20', '--top-k-edges', '30', '--output-json', str(graph_paths['json']), '--output-html', str(graph_paths['html']), '--output-mermaid', str(graph_paths['md'])])
    for suffix in graph_paths:
        shutil.copy2(graph_paths[suffix], drive_graph_paths[suffix])
graph = json.loads(graph_paths['json'].read_text())
positive_features = [node for node in graph['nodes'] if re.fullmatch(r'layer_\d+_feature_\d+', str(node.get('id', ''))) and float(node.get('attribution', 0.0)) > 0]
print('Objective:', graph['attribution_objective'])
print('Objective value:', graph['attribution_objective_value'])
print('Positive graph feature candidates:', len(positive_features))

## 8. Confirm graph-derived carry specificity

Positive graph features are ranked on eight discovery pairs. Frozen cumulative panels are then evaluated on twenty-four fresh carry/no-carry pairs. The matched no-carry intervention distinguishes a carry-specific effect from generic digit-state disruption.

In [ ]:
drive_graph_screen = DRIVE_OUTPUT / 'math_large_10000_topk256_graph_feature_screen.json'
run_cmd([sys.executable, '-m', 'src.math_carry_feature_screen', '--sae-config', CONFIG_ARG, '--addition-csv', DATA_ARG, '--graph', str(graph_paths['json']), '--positions', 'last', '--discovery-cases', '8', '--confirmation-cases', '24', '--seed', '1787', '--exclude-seed', '787', '--exclude-cases', '12', '--panel-sizes', '1', '3', '5', '10', '20', '--primary-panel-size', '10', '--random-panels', '5', '--output', str(drive_graph_screen)])
graph_screen_path = LOCAL_OUTPUT / drive_graph_screen.name
shutil.copy2(drive_graph_screen, graph_screen_path)
graph_screen = json.loads(graph_screen_path.read_text())
graph_primary = graph_screen['confirmation']['primary_result']
print('Graph-panel confirmatory rule passed:', graph_primary['supports_carry_selectivity_under_predeclared_rule'])
display(pd.Series(graph_primary['summary']).to_frame('value'))

## 9. Output-digit-balanced localisation across all SAE latents

This second analysis conditions feature activation on the predicted tens digit before ranking carry differences. Graph-screen cases are excluded. The primary Top-10 panel remains fixed before its confirmation split; Top-20 is reported as secondary hypothesis generation.

In [ ]:
drive_localisation = DRIVE_OUTPUT / 'math_large_10000_topk256_balanced_localisation.json'
drive_cache = DRIVE_OUTPUT / 'math_large_10000_topk256_balanced_activations.npz'
run_cmd([sys.executable, '-m', 'src.math_carry_balanced_localization', '--sae-config', CONFIG_ARG, '--addition-csv', DATA_ARG, '--candidate-pairs', '140', '--discovery-pairs', '32', '--confirmation-pairs', '32', '--seed', '4787', '--batch-size', '8', '--panel-sizes', '1', '3', '5', '10', '20', '--primary-panel-size', '10', '--random-panels', '5', '--exclude-json', str(drive_graph_screen), '--output', str(drive_localisation), '--activation-cache', str(drive_cache)])
localisation_path = LOCAL_OUTPUT / drive_localisation.name
shutil.copy2(drive_localisation, localisation_path)
localisation = json.loads(localisation_path.read_text())
print('Primary Top-10 rule passed:', localisation['primary_result']['supports_compact_carry_selectivity_under_predeclared_rule'])
display(pd.Series(localisation['primary_result']['causal_summary']).to_frame('value'))
top20 = next(panel for panel in localisation['causal_confirmation']['panels'] if panel['name'] == 'top_20')
display(pd.Series(top20['causal_summary']).to_frame('top_20_secondary'))

## 10. One untouched replication, only if Top-20 generates a directional hypothesis

This cell never searches another panel. It runs exactly the frozen Top-20 on thirty-two intervention-untouched cases only when the secondary confirmation effect and its interval are already negative. Otherwise it records that no replication was warranted.

In [ ]:
top20_summary = top20['causal_summary']
top20_ci = top20_summary['bootstrap_95_ci_mean_paired_difference']
replication_warranted = top20_summary['mean_paired_difference'] < 0 and top20_ci[1] < 0
drive_replication = DRIVE_OUTPUT / 'math_large_10000_topk256_top20_replication.json'
if replication_warranted:
    run_cmd([sys.executable, '-m', 'src.math_carry_top20_replication', '--sae-config', CONFIG_ARG, '--source-result', str(drive_localisation), '--replication-pairs', '32', '--seed', '9787', '--output', str(drive_replication)])
    replication_path = LOCAL_OUTPUT / drive_replication.name
    shutil.copy2(drive_replication, replication_path)
    replication = json.loads(replication_path.read_text())
    print('Independent Top-20 replication passed:', replication['primary_result']['replicates_frozen_top20_carry_selectivity'])
    display(pd.Series(replication['primary_result']['causal_summary']).to_frame('value'))
else:
    print('Top-20 did not generate a negative interval; no replication was run.')

## 11. Save provenance and verify persistent artifacts

In [ ]:
for path in LOCAL_OUTPUT.iterdir():
    if path.is_file():
        shutil.copy2(path, DRIVE_OUTPUT / path.name)
provenance = DRIVE_EXPERIMENT / 'provenance'
provenance.mkdir(parents=True, exist_ok=True)
shutil.copy2(DATA_ARG, provenance / Path(DATA_ARG).name)
shutil.copy2(CONFIG_ARG, provenance / Path(CONFIG_ARG).name)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() if (ROOT / '.git').exists() else None
manifest = {'experiment': 'math_large_10000_topk256_position_matched', 'git_commit': commit, 'gpu': torch.cuda.get_device_name(0), 'seed': 787, 'layers': LAYERS, 'corpus_rows': len(corpus), 'carry_counts': corpus.groupby('IsCarry').size().to_dict(), 'train_rows': len(train_indices), 'validation_rows': len(val_indices), 'config': CONFIG_ARG, 'dense_latent_export': False, 'graph_screen': str(drive_graph_screen), 'balanced_localisation': str(drive_localisation), 'top20_replication_warranted': bool(replication_warranted), 'drive_output': str(DRIVE_OUTPUT), 'drive_checkpoints': str(DRIVE_CHECKPOINTS), 'drive_activations': str(DRIVE_ACTIVATIONS)}
manifest_path = LOCAL_OUTPUT / 'experiment_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2))
shutil.copy2(manifest_path, DRIVE_OUTPUT / manifest_path.name)
print(json.dumps(manifest, indent=2))
print('All persistent outputs are under:', DRIVE_OUTPUT)